In [1]:
import os, ray, sys
sys.path.append(".")
from functionsG import *

# === Ray/Modin Environment Variables ===
from init_ray_cluster import init_ray_cluster
# init_ray_cluster()


# === Core Libraries ===
# Import necessary libraries
import numpy as np
from collections import Counter
from collections import defaultdict, deque

# import modin.pandas as mpd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from time import gmtime, localtime, strftime, sleep
from datetime import datetime, timedelta, time

# Import Cox Proportional Hazards Model package lifelines
import lifelines
print(lifelines.__version__)
from lifelines import datasets, CoxPHFitter
from lifelines.utils import to_long_format
import warnings
warnings.filterwarnings('ignore')

import sys, glob, csv
from IPython.display import IFrame, display, HTML


# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")
print("Note: plot_partial_effects_on_outcome is a method of CoxPHFitter, not a separate import")

# Set job_codes to track for visualization
job_code_track_count = 5
print("Job code track count:", job_code_track_count)

# Set the Boolean variable prestige_calc to decide whether or not to do prestige calculations
prestige_calc = True

# === Secure PostgreSQL Connection String and Create Engine ===
pp = run_pp()
from urllib.parse import quote_plus
safe_pw = quote_plus(pp)
params_dict = get_params()
conn_str = f"XXXXXXXXXX"

# SQL and Feather I/O =
# import psycopg2
# from psycopg2 import sql, extras
from sqlalchemy import create_engine, text
# ---2. Create SQLAlchemy Engine ---
engine = create_engine(conn_str)
check_sqlalchemy_connection(engine)

# === Directories ===
var_dir = './running_vars'

def ray_cleanup(object_refs: list = None, verbose: bool = True):
    """
    Frees Ray object references and runs garbage collection.
    Does NOT shut down or re-sart Ray.
    Safe to use between pipeline steps if Ray session should
    rmeain alive.
    """
    import ray, gc
    if object_refs:
        for ref in object_refs:
            try:
                ray.internal.free(ref)
                if verbose:
                    print("Freed Ray object: {ref}")
            except Exception as e:
                if verbose:
                    print(f"Ray cleanup failed for object: {e}")
    if verbose:
        print("Running garbage collection...")
        gc.collect()
    if verbose:
        print("Memory cleanup complete.")  
        
def check_table(table_name, show=True):
    # 
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
        row_count = result.scalar()
        if show:
            print(f"Table {table_name} is accessible and has {row_count:,} rows.")
    return row_count

def table_to_df(table_name,nest=0,modin=True,show=True):
    # === Load the table into DataFrame ===
    row_count = check_table(table_name,show=show)
    action_p = (f"Reading the new table '{table_name}' with {row_count:,} rows into pandas DataFrame.")
    pandy = time_start(action_p,nest=nest+1,show=show)
    try:
        if modin:
            print("Using modin ray...")
            init_ray_cluster()
            import modin.pandas as mpd
            engine = create_engine(conn_str)
            df = mpd.read_sql(F"SELECT * FROM {table_name}",engine)
            import pandas as pd
            df_out = df._to_pandas()
            del df
            ray_cleanup()
            ray.shutdown()
            
        else:
            print("Using regular pandas")
            import pandas as pd
            engine = create_engine(conn_str)
            df_out = pd.read_sql(F"SELECT * FROM {table_name}",engine)
        time_stop(pandy,action = f"To read {row_count:,} rows..",nest=nest+1,show=show)
        return df_out

    except Exception as e:
        time_stop(pandy,action = f"To read {row_count:,} rows..")
        print(f"Failed to read {table_name}: {e}")

def list_compare(list1, list2, list1_name=None, list2_name=None,show=True):
    set1 = set(list1); set2 = set(list2)
    if not list1_name:
        list1_name = 'list 1'
    if not list2_name:
        list2_name = 'list 2'
    if show:
        print(f"There are {len(set1):,} unique items in {list1_name}.")
        print(f"There are {len(set2):,} unique items in {list2_name}.")
    one_not_two = [item for item in set1 if item not in set2]
    if show:
        print(f"There are {len(one_not_two):,} unique items in {list1_name} not in {list2_name}.")
    two_not_one = [item for item in set2 if item not in set1]
    if show:
        print(f"There are {len(two_not_one):,} unique items in {list2_name} not in {list1_name}.")
    one_and_two = [item for item in set1 if item in set2]
    if show:
        print(f"There are {len(one_and_two):,} unique items in {list1_name} intersect {list2_name}.")
    return one_not_two, two_not_one, one_and_two

def cut_pids(df_in,cut_pid_list):
    df_out = df_in[~df_in['pid_pde'].isin(cut_pid_list)]
    return df_out

def update_list(source_list,target_list_name,loc_dir=var_dir,load_method='json'):
    if os.path.exists(os.path.join(loc_dir,target_list_name)):
        print(f"Getting {target_list_name}.....")
        if load_method == 'pickle':
            work_list = load_pickle(target_list_name,loc_dir)
        else:
            work_list = load_json(target_list_name,loc_dir)
        print(f"Updating stored list {target_list_name} with this one...")
        out_list = list(set(source_list+work_list))
        print("Returning combined, unique-element list")
        return out_list
    else:
        print(f"Creating {target_list_name} in {loc_dir}..")
        if load_method == 'pickle':
            store_pickle(out_list,target_list_name,loc_dir)
        else:
            store_json(source_list,target_list_name,loc_dir)


# Here is the PostgreSQL code that I used for 'lu_uic_1to1_snps'

sql = """DROP TABLE IF EXISTS lu_uic_1to1_snps;

        CREATE TABLE lu_uic_1to1_snps AS
        SELECT DISTINCT
        	m.snpsht_dt,
        	m.asg_uic,
        	p.asg_uic_pde
        FROM study_talent_net.mv_fouo_uz_master_ad_army_v3a m 
        JOIN study_talent_net.mv_master_ad_army_qtr_v3a p 
        ON m.pid_pde = p.pid_pde
        AND m.snpsht_dt = p.snpsht_dt
        WHERE m.asg_uic IS NOT NULL AND p.asg_uic_pde IS NOT NULL
        ORDER BY m.asg_uic, m.snpsht_dt;"""
# print_syntax(sql)

# Here is the SQL code for 'create_uic_set_master_pde'
sql2 = """DROP TABLE IF EXISTS uic_set_master_pde;

        CREATE TABLE uic_set_master_pde AS
        SELECT DISTINCT asg_uic_pde
            FROM  study_talent_net.mv_master_ad_army_qtr_v3a
            WHERE asg_uic_pde IS NOT NULL
        ORDER BY asg_uic_pde;
        
        CREATE INDEX idx_asg_uic_pde_on_uic_set_master_pde ON uic_set_master_pde(asg_uic_pde);"""


# Create a master set of all asg_uic_pde's in the big data!
load_usmp = time_start("Loading uic_set_master_pde into DataFrame",nest=6)
df_usmp = table_to_df('uic_set_master_pde',modin=False,nest=6)
time_stop(load_usmp, action = "Loading uic_set_master_pde into DataFrame",nest=6)
uic_set_master_pde = df_usmp.asg_uic_pde.unique().tolist()
store_json(uic_set_master_pde,'uic_set_master_pde',var_dir)

# Create a master set of all asg_uic's in the big data!
load_usm = time_start("Loading uic_set_master into DataFrame",nest=6)
df_usm = table_to_df('uic_set_master',modin=False,nest=6)
time_stop(load_usm, action = "Loading uic_set_master into DataFrame",nest=6)
uic_set_master = df_usm.asg_uic.unique().tolist()
store_json(uic_set_master,'uic_set_master',var_dir)

we = time_start("Building the whole enchilada: df_uic_lookup and uic_lookup_dict", nest=0)

load_lup = time_start("Loading lu_uic_1to1_snps into DataFrame",nest=6)
df_lup = table_to_df('lu_uic_1to1_snps',modin=False,nest=6)
time_stop(load_lup, action = "Loading lu_uic_1to1_snps into DataFrame",nest=6)
load_lupids = time_start("Loading lu_uic_1to1_snps_pids into DataFrame",nest=6)
df_uic_lu_withpids = table_to_df('lu_uic_1to1_snps_pids',nest=6)
time_stop(load_lupids, action = "Loading lu_uic_1to1_snps_pids into DataFrame",nest=6) 

df_lu_trans= df_uic_lu_withpids.copy()

# Now let's investigate to check to see if there are anyinstances where there is more than one 
# Snapshot for any given pid_pde, with respect to 'asg_uic_pde'
h = time_start("Search pid_pde and snapshot duplicates",nest=6)
dtrans0 = df_lu_trans[df_lu_trans.groupby('snpsht_dt')['pid_pde'].transform(lambda x: x.duplicated(keep=False))]
time_stop(h,nest=6)
bad_pids = dtrans0.pid_pde.unique().tolist()
update_list(bad_pids,'bad_pids')
print(f"\nList of pid_pde's with more than one entry for the same snpsht_dt: {bad_pids}")

# So now we will eliminate those bad pid_pde's from our df_lu_with_pids and take another look
print("---> So now we will eliminate those bad pid_pde's from our df_lu_with_pids and take another look.")
rows_before = len(df_lu_trans)
print(f"'df_lu_trans' has {rows_before:,} rows.")
df_lu_trans_pids = len(df_lu_trans.pid_pde.unique().tolist())
print(f"df_lu_trans currently has {df_lu_trans_pids:,} unique pid_pde's.") 
print(f"Cutting (cut_pids()) {len(bad_pids):,} pids with ambiguous UIC assignments")
df_lu_trans = cut_pids(df_lu_trans,bad_pids)
rows_after = len(df_lu_trans)
print(f"df_lu_trans Now has {rows_after:,} rows after dropping {rows_before-rows_after:,} rows.\n")

# Now let's drop asg_uic_pde, then drop_duplicates, 
# Now let's investigate to check to see if there are anyinstances where there is more than one 
# Snapshot for any given pid_pde, with respect to 'asg_uic'
h = time_start("Search pid_pde and snapshot duplicates",nest=6)
dtrans1 = df_lu_trans.drop('asg_uic_pde',axis=1).drop_duplicates()
dtrans2 = dtrans1.groupby('snpsht_dt')['pid_pde'].transform(lambda x: x.duplicated(keep=False))
time_stop(h,nest=6)
dtrans1[dtrans2].pid_pde.nunique()

df_lu = df_lu_trans.copy()
df_lu = cut_pids(df_lu,bad_pids)
display(df_lu.head(5))
print("The imported lookup table ('lu_uic_1to1_snps') has snpsht_dt and pid_pde columns which are artifacts that should now be dropped.  We'll drop them and then drop duplicates.\n")
df_lu = df_lu.drop(['pid_pde','snpsht_dt'],axis=1).drop_duplicates()
display(df_lu.head(5))

# Let's load a working dataframe to investigate how well our lookup dictionary will work
df_analysis = load_feather('df_501_base')
df_analysis = df_analysis.dropna(subset = ['asg_uic_pde'])
base_df_list = df_analysis.asg_uic_pde.unique().tolist()


df_lu = df_lu.drop_duplicates()
print("Just looking at the lu df with keeping the snpsht_dt")
df_lu = df_lu.drop_duplicates()
lu_list = df_lu.asg_uic.unique().tolist(); lu_list_pde = df_lu.asg_uic_pde.unique().tolist()
lu_not_base_df, base_df_not_lu, lu_and_base_df = list_compare(lu_list_pde,base_df_list,'lu_list_pde','base_df_list')

print("\nGetting rid of every asg_uic_pde that occurs more than once in the dataframe")
dup_matches = df_lu.groupby('asg_uic_pde')['asg_uic'].nunique()
no_asg_uic_pde_dups_list = dup_matches[dup_matches == 1].index.tolist()
clean_lu_not_base_df, base_df_not_clean_lu, clean_lu_and_base_df = list_compare(no_asg_uic_pde_dups_list,base_df_list,'no_asg_uic_pde_dups_list','base_df_list')

# Get a list of asg_uic_pdes that are in df_analysis, but not in the 
# lu dataframe that has had it's uics with duplicate matches cleaned out
uic_in_lu_but_not_clean_lu = [uic for uic in base_df_not_clean_lu if uic in lu_list_pde]
# Now get a list of their prefixes
dup_uic_list_prefixes = [uic[:6] for uic in uic_in_lu_but_not_clean_lu]
# Now get a list of df_analysis prefixes
base_df_prefixes = [uic[:6] for uic in base_df_list]

# Now compare the prefixes in the small list eliminated
print("\nNow compare the prefixes in the small list eliminated")
one_not_two, two_not_one, one_and_two = list_compare(dup_uic_list_prefixes,base_df_prefixes,'dup_uic_list_prefixes','base_df_prefixes')

dup_drop_uics = base_df_not_clean_lu.copy()
print(f"""\n
There are {len(dup_drop_uics):,} asg_uic_pdes from the lookup 1-to-1 DataFrame that have duplicate matches, which we
eliminated from the lookup DataFrame.  However, it turns out that of those {len(dup_drop_uics):,}, {len(one_not_two):,} can't still be identified by prefixes!""")

# Store dup_drop_uics to be used lated with prefixes
store_json(dup_drop_uics,'dup_drop_uics',var_dir)

# Create our final Lookup DataFrame to add a column to our base analysis DataFrame
df_uic_lookup = df_lu.dropna().copy()
if df_uic_lookup.asg_uic.nunique() == df_uic_lookup.asg_uic_pde.nunique():
    store_feather(df_uic_lookup,'df_uic_lookup')
    # Build and store a uic_lookup_dict
    uic_lookup_dict = {k:v for k,v in zip(df_uic_lookup['asg_uic_pde'].tolist() + df_uic_lookup['asg_uic'].tolist(),
                                          df_uic_lookup['asg_uic'].tolist() + df_uic_lookup['asg_uic_pde'].tolist())}
    store_json(uic_lookup_dict,'uic_lookup_dict',var_dir)
    print(f"The DataFrame df_uic_lookup, which has {len(df_uic_lookup):,} rows, and the Dictionary uic_lookup_dict are both built and stored!!")
else:
    print(f"PROBLEM: one of the columns of df_uic_lookup has duplicate entries!!!!!!!!!!!!!!!!!!!!")

time_stop(we,action="Building the whole enchilada: df_uic_lookup and uic_lookup_dict",nest=0)